In [7]:
import coiled

import fsspec
import numpy as np
import rioxarray
import xarray as xr
import fsspec
import pandas as pd
import logging
from flox.xarray import xarray_reduce
import numpy as np
import dask

In [8]:
logging.getLogger("distributed.client").setLevel(logging.ERROR)  # or logging.ERROR

In [9]:
fs = fsspec.filesystem("s3", requester_pays=True)

In [ ]:
!aws configure list 

In [10]:
cluster = coiled.Cluster(
    name="tcl_dask",
    region="us-east-1", # close to dataset, avoid egress charges
    n_workers=50,
    tags={"project": "tcl_dask"},
    scheduler_vm_types="r7g.xlarge", # memory optimized AWS EC2 instances
    worker_vm_types="r7g.2xlarge",
    compute_purchase_option="spot_with_fallback"
)

client = cluster.get_client()

Output()

╭──────────────────────────────── Package Info ────────────────────────────────╮
│                ╷                                                             │
│   Package      │ Note                                                        │
│ ╶──────────────┼───────────────────────────────────────────────────────────╴ │
│   flox         │ Wheel built from ~/flox-0.10.3.tar.gz                       │
│                ╵                                                             │
╰──────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────── Not Synced with Cluster ───────────────────────────╮
│                 ╷                                                ╷           │
│   Package       │ Error                                          │ Level     │
│ ╶───────────────┼────────────────────────────────────────────────┼─────────╴ │
│   pygwalker     │ Pip check had the following issues that need   │ Warning   │
│                 │ resolving:                                     │           │
│                 │ pygwalker 0.3.17 has requirement               │           │
│                 │ duckdb==0.9.2, but you have duckdb 1.3.0.      │           │
│                 │ pygwalker 0.3.17 has requirement               │           │
│                 │ segment-analytics-python==2.2.3, but you have  │           │
│                 │ segment-analytics-python 2.3.3.                │           │
│   pydantic_core │ pydantic-core~=2.33.2 has no install candidate │ Warning   │
│                 │ for Python 3.13 linux-aarch64 on conda-forge   │           │
│   awscrt        │ awscrt~=0.26.1 has no install candidate for    │ Warning   │
│                 │ Python 3.13 linux-aarch64 on conda-forge       │           │
│                 ╵                                                ╵           │
╰──────────────────────────────────────────────────────────────────────────────╯

Output()

In [11]:
dist_obj_name = "s3://gfw-data-lake/umd_glad_dist_alerts/v20250510/raster/epsg-4326/zarr/dist_alerts_full.zarr"
ZARR_V3_EXPERIMENTAL_API=1
dist_alerts = xr.open_zarr(dist_obj_name, zarr_format=3).band_data
dist_alerts = dist_alerts.where(dist_alerts > 0)
dist_alerts

<xarray.DataArray 'band_data' (band: 1, y: 480000, x: 1440000)> Size: 3TB
dask.array<where, shape=(1, 480000, 1440000), dtype=float32, chunksize=(1, 10000, 10000), chunktype=numpy.ndarray>
Coordinates:
  * band         (band) int64 8B 1
  * y            (y) float64 4MB 60.0 60.0 60.0 60.0 ... -60.0 -60.0 -60.0 -60.0
    spatial_ref  int64 8B 0
  * x            (x) float64 12MB -180.0 -180.0 -180.0 ... 180.0 180.0 180.0
Attributes:
    AREA_OR_POINT:             Area
    STATISTICS_MAXIMUM:        31547
    STATISTICS_MEAN:           22058.51055569
    STATISTICS_MINIMUM:        20854
    STATISTICS_STDDEV:         2562.7387616052
    STATISTICS_VALID_PERCENT:  0.001545

In [12]:
natural_lands_obj_name = "s3://gfw-data-lake/sbtn_natural_forests_map/v202410/raster/epsg-4326/zarr/natural_lands.zarr"

natural_lands = xr.open_zarr(natural_lands_obj_name).band_data
natural_lands

<xarray.DataArray 'band_data' (band: 1, y: 560000, x: 1440000)> Size: 806GB
dask.array<open_dataset-band_data, shape=(1, 560000, 1440000), dtype=uint8, chunksize=(1, 10000, 10000), chunktype=numpy.ndarray>
Coordinates:
  * band         (band) int64 8B 1
  * x            (x) float64 12MB -180.0 -180.0 -180.0 ... 180.0 180.0 180.0
    spatial_ref  int64 8B ...
  * y            (y) float64 4MB 80.0 80.0 80.0 80.0 ... -60.0 -60.0 -60.0 -60.0
Attributes:
    AREA_OR_POINT:  Area

In [13]:
countries_obj_name = "s3://gfw-data-lake/gadm_administrative_boundaries/v4.1.85/raster/epsg-4326/zarr/adm0.zarr"

countries = xr.open_zarr(countries_obj_name).band_data
countries

<xarray.DataArray 'band_data' (band: 1, y: 720000, x: 1440000)> Size: 1TB
dask.array<open_dataset-band_data, shape=(1, 720000, 1440000), dtype=uint8, chunksize=(1, 10000, 10000), chunktype=numpy.ndarray>
Coordinates:
  * y            (y) float64 6MB 90.0 90.0 90.0 90.0 ... -90.0 -90.0 -90.0 -90.0
  * x            (x) float64 12MB -180.0 -180.0 -180.0 ... 180.0 180.0 180.0
  * band         (band) int64 8B 1
    spatial_ref  int64 8B ...
Attributes:
    AREA_OR_POINT:             Area
    STATISTICS_MAXIMUM:        124
    STATISTICS_MEAN:           124
    STATISTICS_MINIMUM:        124
    STATISTICS_STDDEV:         0
    STATISTICS_VALID_PERCENT:  0.0116

In [14]:
dist_alerts_aligned, natural_lands_aligned, countries_aligned = xr.align(
    dist_alerts, natural_lands, countries, join="left"
)

In [15]:
natural_lands_aligned

<xarray.DataArray 'band_data' (band: 1, y: 480000, x: 1440000)> Size: 691GB
dask.array<getitem, shape=(1, 480000, 1440000), dtype=uint8, chunksize=(1, 10000, 10000), chunktype=numpy.ndarray>
Coordinates:
  * band         (band) int64 8B 1
  * y            (y) float64 4MB 60.0 60.0 60.0 60.0 ... -60.0 -60.0 -60.0 -60.0
  * x            (x) float64 12MB -180.0 -180.0 -180.0 ... 180.0 180.0 180.0
    spatial_ref  int64 8B ...
Attributes:
    AREA_OR_POINT:  Area

In [16]:
%%time

natural_lands.max().compute()

CPU times: user 1.32 s, sys: 92.1 ms, total: 1.42 s
Wall time: 13.1 s


<xarray.DataArray 'band_data' ()> Size: 1B
array(2, dtype=uint8)
Coordinates:
    spatial_ref  int64 8B 0

In [17]:
%%time
natural_lands_aligned.max().compute()

CPU times: user 1.58 s, sys: 6.97 ms, total: 1.58 s
Wall time: 14.6 s


<xarray.DataArray 'band_data' ()> Size: 1B
array(2, dtype=uint8)
Coordinates:
    spatial_ref  int64 8B 0

In [18]:
natural_lands_aligned.name = "natural_lands"
countries_aligned.name = "countries"

In [19]:
%%time

alerts_count_align = xarray_reduce(
    dist_alerts_aligned,
    *(countries_aligned, natural_lands_aligned),
    func='count',
    expected_groups=(np.arange(255), [0, 1, 2]) 
).compute()

CPU times: user 3.39 s, sys: 103 ms, total: 3.49 s
Wall time: 2min 22s


In [22]:
countries_from_clipped = xr.open_zarr(
    's3://gfw-data-lake/gadm_administrative_boundaries/v4.1.85/raster/epsg-4326/zarr/adm0_clipped_to_dist.zarr'
).band_data
countries_from_clipped

<xarray.DataArray 'band_data' (band: 1, y: 480000, x: 1440000)> Size: 1TB
dask.array<open_dataset-band_data, shape=(1, 480000, 1440000), dtype=uint16, chunksize=(1, 10000, 10000), chunktype=numpy.ndarray>
Coordinates:
  * band         (band) int64 8B 1
  * y            (y) float64 4MB 60.0 60.0 60.0 60.0 ... -60.0 -60.0 -60.0 -60.0
    spatial_ref  int64 8B ...
  * x            (x) float64 12MB -180.0 -180.0 -180.0 ... 180.0 180.0 180.0
Attributes:
    AREA_OR_POINT:             Area
    STATISTICS_MAXIMUM:        124
    STATISTICS_MEAN:           124
    STATISTICS_MINIMUM:        124
    STATISTICS_STDDEV:         0
    STATISTICS_VALID_PERCENT:  0.0116

In [23]:
regions_from_clipped = xr.open_zarr(
    's3://gfw-data-lake/gadm_administrative_boundaries/v4.1.85/raster/epsg-4326/zarr/adm1_clipped_to_dist.zarr'
).band_data
regions_from_clipped

<xarray.DataArray 'band_data' (band: 1, y: 480000, x: 1440000)> Size: 691GB
dask.array<open_dataset-band_data, shape=(1, 480000, 1440000), dtype=uint8, chunksize=(1, 10000, 10000), chunktype=numpy.ndarray>
Coordinates:
  * band         (band) int64 8B 1
  * x            (x) float64 12MB -180.0 -180.0 -180.0 ... 180.0 180.0 180.0
    spatial_ref  int64 8B ...
  * y            (y) float64 4MB 60.0 60.0 60.0 60.0 ... -60.0 -60.0 -60.0 -60.0
Attributes:
    AREA_OR_POINT:             Area
    STATISTICS_MAXIMUM:        8
    STATISTICS_MEAN:           8
    STATISTICS_MINIMUM:        8
    STATISTICS_STDDEV:         0
    STATISTICS_VALID_PERCENT:  0.0116

In [24]:
subregions_from_clipped = xr.open_zarr(
    's3://gfw-data-lake/gadm_administrative_boundaries/v4.1.85/raster/epsg-4326/zarr/adm2_clipped_to_dist.zarr'
).band_data

In [25]:
natural_lands_from_clipped  = xr.open_zarr(
    's3://gfw-data-lake/sbtn_natural_forests_map/v202410/raster/epsg-4326/zarr/clipped_to_dist.zarr'
).band_data

In [26]:
pixel_area = xr.open_zarr(
    's3://gfw-data-lake/umd_area_2013/v1.10/raster/epsg-4326/zarr/pixel_area_clipped_to_dist.zarr'
).band_data

In [27]:
adm0_ids = [
    0, 4, 8, 10, 12, 16, 20, 24, 28, 31, 32, 36, 40, 44, 48, 50, 51, 52, 56, 60,
    64, 68, 70, 72, 74, 76, 84, 86, 90, 92, 96, 100, 104, 108, 112, 116, 120,
    124, 132, 136, 140, 144, 148, 152, 156, 158, 162, 166, 170, 174, 175, 178,
    180, 184, 188, 191, 192, 196, 203, 204, 208, 212, 214, 218, 222, 226, 231,
    232, 233, 234, 238, 239, 242, 246, 248, 250, 254, 258, 260, 262, 266, 268,
    270, 275, 276, 288, 292, 296, 300, 304, 308, 312, 316, 320, 324, 328, 332,
    334, 336, 340, 344, 348, 352, 356, 360, 364, 368, 372, 376, 380, 384, 388,
    392, 398, 400, 404, 408, 410, 414, 417, 418, 422, 426, 428, 430, 434, 438,
    440, 442, 446, 450, 454, 458, 462, 466, 470, 474, 478, 480, 484, 492, 496,
    498, 499, 500, 504, 508, 512, 516, 520, 524, 528, 531, 533, 534, 535, 540,
    548, 554, 558, 562, 566, 570, 574, 578, 580, 581, 583, 584, 585, 586, 591,
    598, 600, 604, 608, 612, 616, 620, 624, 626, 630, 634, 638, 642, 643, 646,
    652, 654, 659, 660, 662, 663, 666, 670, 674, 678, 682, 686, 688, 690, 694,
    702, 703, 704, 705, 706, 710, 716, 724, 728, 729, 732, 740, 744, 748, 752,
    756, 760, 762, 764, 768, 772, 776, 780, 784, 788, 792, 795, 796, 798, 800,
    804, 807, 818, 826, 831, 832, 833, 834, 840, 850, 854, 858, 860, 862, 876,
    882, 887, 894
]


In [28]:
%%time

from flox import ReindexArrayType, ReindexStrategy


countries_from_clipped.name = "countries"
regions_from_clipped.name = "regions"
subregions_from_clipped.name = "subregions"
natural_lands_from_clipped.name = "natural_lands"
alerts_count = xarray_reduce(
    pixel_area,
    # *(countries_from_clipped, regions_from_clipped, subregions_from_clipped, natural_lands_from_clipped),
    countries_from_clipped,
    func='count',
    # expected_groups=(adm0_ids, np.arange(86), np.arange(854), [0, 1, 2]),
    expected_groups=adm0_ids,
    reindex=ReindexStrategy(
        blockwise=False, array_type=ReindexArrayType.SPARSE_COO
    ),
    fill_value=0
).compute()

CPU times: user 1.41 s, sys: 52.2 ms, total: 1.46 s
Wall time: 1min 30s


In [29]:
sparse_data = alerts_count.data

dim_names = alerts_count.dims
indices = sparse_data.coords  # tuple of arrays with indices into each dim
values = sparse_data.data     # non-zero values

# Step 4: Map dimension indices to coordinate values
coord_dict = {
    dim: alerts_count.coords[dim].values[indices[i]]
    for i, dim in enumerate(dim_names)
}
coord_dict["value"] = values

df = pd.DataFrame(coord_dict)

In [30]:
df.to_parquet('./dist_alerts_by_natural_lands.parquet', index=False)

In [33]:
df[(df.countries == 566)].sort_values(by='value').head(50)

,countries,value
157,566,1198042769


In [34]:
df[(df.countries == 204)].sort_values(by='value').head(50)

,countries,value
59,204,152004491
